# 문제 2-1 : 콤마 구분 리스트 파서 활용 — 한국 명소·활동 추천기

관심 분야(예: "음식", "스포츠", "영화")를 입력하면, 그 분야와 관련된 **한국의 유명한 장소나 활동 5가지**를 콤마로 구분된 리스트로 출력한다.

**출력 파서**: `CommaSeparatedListOutputParser`

**요구사항 체크**
- [x] `CommaSeparatedListOutputParser` 사용
- [x] 사용자 입력을 받아 동적으로 처리 (섹션 7)
- [x] 한국 관련 내용으로 추천 (프롬프트에서 한국으로 한정)

## 1. 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정 (.env 는 프로젝트 루트에 있음)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 전체는 출력하지 않고 설정 여부만 확인
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

## 2. LLM 설정 (ChatOpenAI)

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq"(무료) 또는 "openai" — 이 값만 바꾸면 아래 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# Groq는 OpenAI 호환 API라서 ChatOpenAI에 base_url만 바꿔서 사용
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}


def get_llm(temperature: float = 0.7, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자의 ChatOpenAI 인스턴스를 생성합니다."""
    config = LLM_CONFIGS[provider or PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {(provider or PROVIDER).upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, max_retries=3)


llm = get_llm()
print(f"{PROVIDER} / {llm.model_name}")

## 3. 출력 파서 생성 — `CommaSeparatedListOutputParser`
- `get_format_instructions()` : "콤마로 구분해서 답하라"는 형식 지시문을 만들어 준다 → 프롬프트에 넣는다
- `parse()` : LLM의 문자열 답변을 파이썬 `list` 로 바꿔 준다

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

# 프롬프트에 넣을 형식 지시문 (파서가 자동으로 만들어 줌)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

# parse() 동작 확인: 콤마로 구분된 문자열 → 리스트
print(output_parser.parse("명동 칼국수, 부산 돼지국밥, 전주 비빔밥"))

## 4. PromptTemplate 작성
`{field}` 자리에 사용자가 입력한 분야가 들어가고, `{format_instructions}` 는 `partial_variables` 로 미리 채워 둔다.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """당신은 한국의 여행·문화 전문가입니다.
사용자가 관심 있는 분야와 관련된 한국의 유명한 장소나 활동 5가지를 추천하세요.

규칙:
- 반드시 한국(대한민국)에 있는 장소나 한국에서 할 수 있는 활동만 추천
- 정확히 5개
- 각 항목은 '지역명 + 장소/음식/활동' 형태의 짧은 명사구 (예: 전주 비빔밥, 잠실야구장 직관)
- 번호, 설명, 따옴표 없이 항목만 콤마로 구분해서 한 줄로 답변

관심 분야: {field}

{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["field"],
    partial_variables={"format_instructions": format_instructions},
)

print(prompt.format(field="음식"))

## 5. LCEL 체인 연결: `prompt | llm | output_parser` — 예시 입력 `"음식"`

In [ ]:
chain = prompt | llm | output_parser

result = chain.invoke({"field": "음식"})

print(type(result))  # CommaSeparatedListOutputParser 덕분에 str 이 아니라 list
print(f"항목 수: {len(result)}")
print(result)

## 6. 여러 분야로 테스트 — `"음식"`, `"스포츠"`, `"영화"`

In [ ]:
def recommend_korea(field: str) -> list[str]:
    """관심 분야를 받아 한국의 유명한 장소/활동 5가지를 리스트로 반환"""
    items = chain.invoke({"field": field})
    print(f"입력: \"{field}\"")
    print(f"출력: {items}")
    if len(items) != 5:
        print(f"  ⚠️ 5개가 아니라 {len(items)}개가 나왔습니다.")
    print("-" * 60)
    return items


for field in ["음식", "스포츠", "영화"]:
    recommend_korea(field)

## 7. 사용자 입력을 받아 동적으로 처리
분야를 바꿔 가며 계속 입력할 수 있다. 아무것도 입력하지 않거나 `종료` 를 입력하면 끝난다.

In [ ]:
while True:
    field = input("관심 분야를 입력하세요 (끝내려면 엔터 또는 '종료'): ").strip()
    if field in ("", "종료"):
        print("프로그램을 종료합니다.")
        break
    recommend_korea(field)